Comentarios a la práctica obligatoria Unidad 2 ML Supervisado Clasificación con Regresión Logística

In [ ]:
# generar una variable con una lista de las columnas a excluir
excluidas = ["embarked","class","adult_male", "alive"]

Ver y tratar datos faltantes de una columna ( en este caso df_titanic.deck)

In [ ]:
# Ver datos faltantes de una columna
df["deck"].value_counts(dropna=False, normalize= True)

In [ ]:
# Tiene más de un 77% de Nan, podemos borrar la columna,
# o dar etiqueta de desconocido ("UNK" = unknown). Va a depender de la importancia de la columna para el negocio.
# En el segundo caso: Creamos una copia del DataFrame y rellenamos
df_deck = df.copy()
df_deck["deck"] = df_deck.deck.fillna("UNK")

# SOLO PODEMOS HACERLO CON CATEGÓRICAS
# ¿Es una imputación? Debate, no cambia las proporciones de los valores, puede que no sea una imputación.

In [ ]:
# Creamos dos posibles grupo de features, una que contenga deck y otra que no. 
# Dejamos abierta las dos posibilidades.

features_base = [col for col in df_deck.columns if col not in excluidas]
features_base.remove("deck")

features_base_deck= [col for col in df_deck.columns if col not in excluidas]

In [ ]:
# En embark_town tenemos datos faltantes, he borrado las filas correspondientes a esos datos.
# Otra opción es rellenar con la moda, la categoría que más se repite:
df_deck.loc[df_deck["embark_town"].isna(),"embark_town"] = df_deck["embark_town"].mode()[0]

Hacemos el split con stratify por embark_town, y luego comprobamos que haya estratificado bien.  

#### Imputamos nulos de las numéricas en base a valores de **train**

Yo he hecho las medias de edades por clase, se puede hacer por otra variable como "who", mujeres,hombres y niñ@s.    

En este caso, solo nos quedan nulos en la columna "age". Al tratarse de la variable de edad, y dado que tenemos pasajeros de diversas edades, si decidimos imputar por la mediana de la variable podrían darse casos en los que imputemos edades de 28 años a niños, lo que evidentemente no tiene sentido.

Un enfoque más fino es imputar sus respectivas medias o medianas en función del genero, es decir, imputar a hombres, mujeres y niñ@s sus respectivas medias medias o medianas.

En nuestro ejemplo, elegimos la mediana no solo por ser más resistente a valores atípicos sino porque nos permite imputar variables discretas.

In [ ]:
# Elegimos la mediana no tanto por la distribución de edades sino para imputar variables discretas.

es_hombre = train_set.who == "man"
es_mujer = train_set.who == "woman"
es_child = train_set.who == "child"

median_man = train_set[es_hombre]["age"].median()
median_woman = train_set[es_mujer]["age"].median()
median_child = train_set[es_child]["age"].median()

es_nulo = train_set.age.isna()
es_nulo_test = test_set.age.isna()

#Imputamos en train
train_set.loc[es_hombre & es_nulo, "age"] = median_man
train_set.loc[es_mujer & es_nulo, "age"] = median_woman
train_set.loc[es_child & es_nulo, "age"] = median_child

#Imputamos en test
test_set.loc[(test_set.who == "man") & es_nulo_test, "age"] = median_man
test_set.loc[(test_set.who == "woman") & es_nulo_test, "age"] = median_woman
test_set.loc[(test_set.who == "child") & es_nulo_test, "age"] = median_child

TODO LO ANTERIOR LO HEMOS HECHO ANTES DEL MINI-EDA

Miramos la distribución del target, survived. Está deseqilibrado, no sobrevive 63, sobrevive 37. Esto significa que si no lo equilibramos, el modelo podrá predecir mejor los no sobrevive que los sobrevive. **Es lo que ha pasado en mi ejercicio**.

Normalización de variables numéricas.  
No podemos hacer el logaritmo, otras posibles opciones:  
1. Hacer log(x+1)  Mi elección
2. Hacer sqrt(x) o cbrt(x)  (raiz cúbica)  

LAS Métricas:  
Muy importante, entender las métricas en función del tipo de problema:  
 Métricas de regresión (MAE, MSE, RMSE, MAPE, R^2)  
 Métricas de clasificación (accuracy, precision, recall, f1-score, auroc)  

¡OJO!  
CUIDADO CON LA REDUNDANCIA DE VARIABLES.  
